# LeetCode 320 — Generalized Abbreviation bằng Python

Link đề: https://leetcode.com/problems/generalized-abbreviation/ · Độ khó: **Medium** · Chủ đề: String, Backtracking, Bit Manipulation

> Đây là bài **Premium**: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def generateAbbreviations(self, word: str) -> List[str]:
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **viết tắt tổng quát** của một từ là cách thay thế **một số đoạn con không kề nhau** (có thể rỗng) bằng **độ dài** của đoạn đó. Ví dụ `"word"` → `"w2d"`, `"1o1d"`, `"4"`. Cần liệt kê **mọi** cách viết tắt (thứ tự trả về bất kỳ).
2. Nhận ra **mỗi vị trí chỉ có hai lựa chọn**: **giữ nguyên chữ** hoặc **gộp vào một số đếm**, nên tổng số kết quả là **`2ⁿ`**.
3. Đi từ **vét cạn bằng bitmask (`mask` cho biết vị trí nào bị viết tắt)** → **quay lui (backtracking) có đếm số chữ đang bị gộp** → **đệ quy trả về danh sách của phần đuôi** → **sinh lặp bằng danh sách trạng thái `(chuỗi, đếm)`** → **chia để trị (nối kết quả hai nửa, gộp hai số kề nhau)**.
4. Hiểu **quy tắc không có hai số kề nhau**: số đếm phải được **chốt** (đưa vào chuỗi) ngay trước khi gặp một chữ được giữ lại và khi hết từ.
5. Kiểm thử bài **nhiều đáp án / thứ tự tùy ý**: so sánh **danh sách đã sắp xếp**, kiểm tra **mỗi chuỗi hợp lệ** (hàm kiểm tra viết tắt), **đôi một khác nhau**, và **số lượng đúng `2ⁿ`**.

## 2. Đề bài & ví dụ

Viết tắt tổng quát của một từ có thể được tạo bằng cách thay **số chữ cái bất kỳ (kể cả không)** bằng **số lượng chữ cái được thay**, với điều kiện các đoạn bị thay **không liền kề nhau** (hai đoạn kề nhau phải được gộp thành một).

Cho một từ `word`, trả về **danh sách mọi cách viết tắt tổng quát** của `word`. Có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1:** `word = "word"` → `["4","3d","2r1","2rd","1o2","1o1d","1or1","1ord","w3","w2d","w1r1","w1rd","wo2","wo1d","wor1","word"]` (16 chuỗi).

**Ví dụ 2:** `word = "a"` → `["1","a"]`.

**Ràng buộc:** `1 <= word.length <= 15`; `word` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import random
import re
from itertools import combinations
from math import comb

example_word = "word"
example_expected = ["4", "3d", "2r1", "2rd", "1o2", "1o1d", "1or1", "1ord",
                    "w3", "w2d", "w1r1", "w1rd", "wo2", "wo1d", "wor1", "word"]
examples = [
    ("word", sorted(example_expected)),
    ("a", sorted(["1", "a"])),
]
for w, e in examples:
    print("word =", repr(w), "-> kỳ vọng (đã sắp)", e)

word = 'word' -> kỳ vọng (đã sắp) ['1o1d', '1o2', '1or1', '1ord', '2r1', '2rd', '3d', '4', 'w1r1', 'w1rd', 'w2d', 'w3', 'wo1d', 'wo2', 'wor1', 'word']
word = 'a' -> kỳ vọng (đã sắp) ['1', 'a']


## 3. Kiến thức nền

### 3.1. Mỗi vị trí hai lựa chọn: "giữ" hoặc "gộp"

Với từ dài `n`, gọi `mask` là một số `n` bit: bit `i` bằng `1` nghĩa là chữ `word[i]` **bị viết tắt** (gộp vào số đếm), bằng `0` nghĩa là **giữ**. Một `mask` xác định đúng một cách viết tắt: mỗi **đoạn liên tiếp các bit 1** thành **một số** (độ dài đoạn); các bit 0 giữ lại chữ. Có `2ⁿ` mặt nạ, nên có `2ⁿ` cách viết tắt, đôi một khác nhau.

In [2]:
def abbreviation_from_mask(word, mask):
    n = len(word)
    out = []
    count = 0
    for i in range(n):
        if (mask >> i) & 1:
            count += 1                              # chữ này bị gộp vào số đếm
        else:
            if count:
                out.append(str(count))               # chốt số đếm trước khi gặp chữ được giữ
                count = 0
            out.append(word[i])
    if count:
        out.append(str(count))                       # chốt số đếm ở cuối từ
    return "".join(out)


for mask in (0b0000, 0b1111, 0b1000, 0b0110, 0b0101):
    print(f"mask = {mask:04b}  ->  {abbreviation_from_mask('word', mask)}")

mask = 0000  ->  word
mask = 1111  ->  4
mask = 1000  ->  wor1
mask = 0110  ->  w2d
mask = 0101  ->  1o1d


### 3.2. Hai số không được kề nhau

Nếu viết `1` rồi `1` thành `"11"` thì **không** thể phân biệt với số mười một. Vì vậy hai đoạn bị gộp **kề nhau** phải hợp thành **một** đoạn (số cộng lại). Cách dựng ở trên (chỉ chốt số khi gặp chữ được giữ hoặc hết từ) luôn đảm bảo điều đó.

### 3.3. Hàm kiểm tra một chuỗi có phải viết tắt hợp lệ của `word`

Quét đồng thời `word` và `abbr`: gặp chữ thì phải khớp chữ trong `word`; gặp chữ số thì đọc **cả số** (không bắt đầu bằng `0`) rồi nhảy con trỏ `word` tương ứng. Hợp lệ khi cả hai kết thúc đúng lúc. (Đây là bài 408 **Valid Word Abbreviation**.)

In [3]:
def is_valid_abbr(word: str, abbr: str) -> bool:
    i = j = 0
    n, m = len(word), len(abbr)
    while i < n and j < m:
        if abbr[j].isdigit():
            if abbr[j] == "0":
                return False                          # số không được bắt đầu bằng 0
            k = j
            while k < m and abbr[k].isdigit():
                k += 1
            i += int(abbr[j:k])
            j = k
        else:
            if word[i] != abbr[j]:
                return False
            i += 1
            j += 1
    return i == n and j == m


print(is_valid_abbr("word", "w2d"), is_valid_abbr("word", "1o1d"), is_valid_abbr("word", "11"), is_valid_abbr("word", "w3d"), is_valid_abbr("word", "wo01"))

True True False False False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`word = "abc"` có `2³ = 8` cách:

| `mask` (bit 2..0) | các chữ bị gộp | kết quả |
|---|---|---|
| `000` | không | `abc` |
| `001` | `a` | `1bc` |
| `010` | `b` | `a1c` |
| `011` | `a, b` (kề nhau → gộp thành 2) | `2c` |
| `100` | `c` | `ab1` |
| `101` | `a` và `c` | `1b1` |
| `110` | `b, c` | `a2` |
| `111` | cả ba | `3` |

Cách quay lui: ở mỗi vị trí chọn **giữ** (nhớ chốt số đếm trước) hoặc **gộp** (tăng bộ đếm); hết từ thì chốt số còn lại.

In [4]:
print(sorted(abbreviation_from_mask("abc", m) for m in range(8)))

['1b1', '1bc', '2c', '3', 'a1c', 'a2', 'ab1', 'abc']


## 5. Cách 1 — Brute Force / baseline: duyệt mọi `mask` từ `0` đến `2ⁿ − 1`

**Ý tưởng.** Với mỗi `mask`, dựng chuỗi viết tắt bằng hàm `abbreviation_from_mask`. Đơn giản và chắc chắn, `O(n · 2ⁿ)`.

In [5]:
def generate_bitmask(word: str) -> List[str]:
    n = len(word)
    return [abbreviation_from_mask(word, mask) for mask in range(1 << n)]


for w, e in examples:
    assert sorted(generate_bitmask(w)) == e
print("PASS ví dụ (bitmask)")

PASS ví dụ (bitmask)


**Độ phức tạp (Cách 1).** Time `O(n · 2ⁿ)` (với `n = 15`: `15 · 32768 ≈ 5·10⁵` thao tác ký tự); Space `O(n · 2ⁿ)` cho đầu ra (không tính được là thừa vì phải trả đủ `2ⁿ` chuỗi).

**Nút thắt.** Mỗi chuỗi dựng lại từ đầu, không dùng lại phần chung. Quay lui chia sẻ tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

Quay lui theo vị trí `pos` với hai thông tin: **chuỗi đã dựng** `cur` và **số chữ đang bị gộp liên tiếp** `count`. Tại `pos`:

1. **Gộp** `word[pos]`: gọi tiếp với `count + 1` (chưa ghi gì vào `cur`).
2. **Giữ** `word[pos]`: ghi `cur + (str(count) nếu count > 0) + word[pos]`, đặt `count = 0`.

Hết từ (`pos == n`): ghi `cur + (str(count) nếu count > 0)`. Số lá của cây là `2ⁿ`, mỗi lá tương ứng một viết tắt.

Điểm mấu chốt: **chỉ chốt số đếm khi gặp chữ được giữ hoặc khi hết từ**; nhờ đó không bao giờ có hai số kề nhau.

In [6]:
# Số lá của cây quay lui = 2^n
def count_leaves(n):
    leaves = 0
    def go(pos):
        nonlocal leaves
        if pos == n:
            leaves += 1
            return
        go(pos + 1)
        go(pos + 1)
    go(0)
    return leaves


for n in range(0, 11):
    assert count_leaves(n) == 2 ** n
print("PASS: số lá của cây quay lui bằng 2^n (n = 0..10)")

PASS: số lá của cây quay lui bằng 2^n (n = 0..10)


## 7. Các cách tiếp theo

### Cách 2 — Quay lui với `(pos, cur, count)`

**Giải thích.** Đúng như mục 6: hàm `dfs(pos, cur, count)` đẩy kết quả vào danh sách khi `pos == n`.

In [7]:
def generate_backtrack(word: str) -> List[str]:
    n = len(word)
    result = []

    def dfs(pos, cur, count):
        if pos == n:
            result.append(cur + (str(count) if count else ""))
            return
        dfs(pos + 1, cur, count + 1)                                  # gộp word[pos] vào số đếm
        dfs(pos + 1, cur + (str(count) if count else "") + word[pos], 0)   # giữ word[pos]

    dfs(0, "", 0)
    return result


for w, e in examples:
    assert sorted(generate_backtrack(w)) == e
print("PASS ví dụ (quay lui)")

PASS ví dụ (quay lui)


**Độ phức tạp Cách 2:** Time `O(n · 2ⁿ)` (mỗi lá nối chuỗi `O(n)`, nhưng tổng số nút là `2^{n+1}`); Space `O(n)` cho ngăn xếp đệ quy cộng đầu ra.

### Cách 3 — Đệ quy trả về danh sách của phần đuôi (cách tham khảo)

**Giải thích.** `dfs(i)` = mọi viết tắt của `word[i:]`. Với `i`, chọn độ dài `L ≥ 0` của đoạn bị gộp bắt đầu tại `i`: nếu `i + L == n` thì kết quả chỉ là `str(L)` (hoặc `""` khi `L = 0`); ngược lại phải **giữ** chữ `word[i + L]` ngay sau đoạn gộp, rồi nối với `dfs(i + L + 1)`. Điều này đảm bảo hai số không kề nhau.

In [8]:
def generate_suffix(word: str) -> List[str]:
    n = len(word)

    def dfs(i):
        if i == n:
            return [""]
        res = []
        for j in range(i, n + 1):                                # j - i chữ bị gộp, rồi giữ word[j] (nếu j < n)
            prefix = str(j - i) if j > i else ""
            if j == n:
                res.append(prefix)
            else:
                for tail in dfs(j + 1):
                    res.append(prefix + word[j] + tail)
        return res

    return dfs(0)


for w, e in examples:
    assert sorted(generate_suffix(w)) == e
print("PASS ví dụ (đệ quy phần đuôi)")

PASS ví dụ (đệ quy phần đuôi)


**Độ phức tạp Cách 3:** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)` cho các danh sách trung gian của đuôi (có thể ghi nhớ để dùng lại, nhưng không cần).

### Cách 4 — Sinh lặp bằng danh sách trạng thái `(chuỗi, đếm)`

**Giải thích.** Không đệ quy: giữ danh sách các cặp `(cur, count)`. Với mỗi chữ `ch`: tạo danh sách mới gồm `(cur, count + 1)` (gộp) và `(cur + số + ch, 0)` (giữ). Cuối cùng chốt số còn lại.

In [9]:
def generate_iterative(word: str) -> List[str]:
    states = [("", 0)]
    for ch in word:
        nxt = []
        for cur, count in states:
            nxt.append((cur, count + 1))                                   # gộp
            nxt.append((cur + (str(count) if count else "") + ch, 0))       # giữ
        states = nxt
    return [cur + (str(count) if count else "") for cur, count in states]


for w, e in examples:
    assert sorted(generate_iterative(w)) == e
print("PASS ví dụ (sinh lặp)")

PASS ví dụ (sinh lặp)


**Độ phức tạp Cách 4:** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)` (cần giữ mọi trạng thái trung gian).

### Cách 5 — Chia để trị: ghép kết quả hai nửa (gộp số kề nhau)

**Giải thích.** Biểu diễn một viết tắt bằng **dãy token** (chữ cái hoặc số nguyên). `dc(lo, hi)` trả mọi dãy token của `word[lo:hi]`. Với một chữ: `("c",)` hoặc `(1,)`. Với đoạn dài hơn: chia đôi, lấy tích Descartes `L × R`; nếu token cuối của `L` **và** token đầu của `R` đều là số thì **cộng** chúng thành một số. Cuối cùng ghép token thành chuỗi.

In [10]:
def generate_divide(word: str) -> List[str]:
    def dc(lo, hi):
        if hi - lo == 1:
            return [(word[lo],), (1,)]
        mid = (lo + hi) // 2
        out = []
        for left in dc(lo, mid):
            for right in dc(mid, hi):
                if isinstance(left[-1], int) and isinstance(right[0], int):
                    out.append(left[:-1] + (left[-1] + right[0],) + right[1:])      # hai số kề nhau -> cộng
                else:
                    out.append(left + right)
        return out

    return ["".join(str(t) for t in tokens) for tokens in dc(0, len(word))]


for w, e in examples:
    assert sorted(generate_divide(w)) == e
print("PASS ví dụ (chia để trị)")

PASS ví dụ (chia để trị)


**Độ phức tạp Cách 5:** Time `O(n · 2ⁿ)` (tích Descartes sinh đúng `2ⁿ` dãy cho mỗi mức); Space `O(n · 2ⁿ)`.

> Cách 2 là cách nộp bài (quay lui chuẩn). Cách 3 là cách tham khảo. Cách 1 và 4 dễ viết. Cách 5 minh họa chia để trị. Mọi cách đều phải sinh `2ⁿ` chuỗi nên không có cách nào nhanh hơn `Ω(2ⁿ)`.

## 8. TRACE từng bước

Theo dõi quay lui (Cách 2) trên `word = "ab"` (4 lá): in mỗi lời gọi `dfs(pos, cur, count)`.

In [11]:
def trace_backtrack(word):
    n = len(word)
    result = []

    def dfs(pos, cur, count, depth):
        pad = "  " * depth
        if pos == n:
            final = cur + (str(count) if count else "")
            print(f"{pad}pos={pos} cur={cur!r} count={count} -> kết quả {final!r}")
            result.append(final)
            return
        print(f"{pad}pos={pos} cur={cur!r} count={count}: xét {word[pos]!r}")
        print(f"{pad} - gộp:")
        dfs(pos + 1, cur, count + 1, depth + 1)
        print(f"{pad} - giữ:")
        dfs(pos + 1, cur + (str(count) if count else "") + word[pos], 0, depth + 1)

    dfs(0, "", 0, 0)
    return result


assert sorted(trace_backtrack("ab")) == ["1b", "2", "a1", "ab"]

pos=0 cur='' count=0: xét 'a'
 - gộp:
  pos=1 cur='' count=1: xét 'b'
   - gộp:
    pos=2 cur='' count=2 -> kết quả '2'
   - giữ:
    pos=2 cur='1b' count=0 -> kết quả '1b'
 - giữ:
  pos=1 cur='a' count=0: xét 'b'
   - gộp:
    pos=2 cur='a' count=1 -> kết quả 'a1'
   - giữ:
    pos=2 cur='ab' count=0 -> kết quả 'ab'


Bốn lá theo thứ tự: gộp-gộp → `"2"`; gộp-giữ → `"1b"` (chốt số `1` rồi ghi `b`); giữ-gộp → `"a1"` (cuối từ chốt số `1`); giữ-giữ → `"ab"`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (quay lui `dfs(pos, cur, count)`), đúng chữ ký của LeetCode.

In [12]:
class Solution:
    def generateAbbreviations(self, word: str) -> List[str]:
        n = len(word)
        res = []

        def dfs(i, cur, cnt):
            if i == n:
                res.append(cur + (str(cnt) if cnt else ""))
                return
            dfs(i + 1, cur, cnt + 1)
            dfs(i + 1, cur + (str(cnt) if cnt else "") + word[i], 0)

        dfs(0, "", 0)
        return res


for w, e in examples:
    assert sorted(Solution().generateAbbreviations(w)) == e
print("PASS ví dụ đề cho")

PASS ví dụ đề cho


## 10. Vì sao thuật toán đúng

- **Song ánh `mask ↔ viết tắt`:** mỗi viết tắt hợp lệ xác định duy nhất tập vị trí bị gộp (đọc từng số rồi nhảy qua đúng ngần đó chữ), và mỗi tập vị trí cho đúng một viết tắt (các đoạn liên tiếp gộp thành một số). Vậy số viết tắt đúng bằng số tập con, tức **`2ⁿ`**, và chúng đôi một khác nhau.
- **Quay lui đầy đủ:** tại mỗi vị trí đi cả hai nhánh (gộp / giữ), nên duyệt hết `2ⁿ` mặt nạ.
- **Không hai số kề nhau:** số đếm `count` chỉ được ghi vào `cur` ngay trước một chữ được giữ (nên theo sau là chữ cái) hoặc ở cuối từ (không còn gì sau). Do đó giữa hai số bao giờ cũng có ít nhất một chữ cái.
- **Số đếm đúng:** `count` bằng độ dài đoạn liên tiếp các chữ bị gộp vừa qua; nó được đặt lại `0` ngay khi chốt.
- **Kiểm thử đủ/không thừa:** nếu mọi đầu ra đều hợp lệ (`is_valid_abbr`), đôi một khác nhau, và có đúng `2ⁿ` chuỗi thì tập đầu ra **chính là** tập mọi viết tắt (vì tập đó đúng `2ⁿ` phần tử).

## 11. Phân tích Time / Space Complexity

- Số kết quả luôn là `2ⁿ`, mỗi chuỗi dài tối đa `n` ký tự, nên bất kỳ thuật toán nào cũng cần ít nhất `Ω(2ⁿ)` (và thực tế `Ω(n · 2ⁿ)` ký tự đầu ra, với độ dài trung bình ~ `n/2 + ...`).
- **Cách 1 (bitmask):** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)` đầu ra.
- **Cách 2 (quay lui):** Time `O(n · 2ⁿ)`; Space ngoài đầu ra `O(n)` (ngăn xếp đệ quy).
- **Cách 3 (đuôi):** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)` cho danh sách trung gian.
- **Cách 4 (lặp):** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)`.
- **Cách 5 (chia để trị):** Time `O(n · 2ⁿ)`; Space `O(n · 2ⁿ)`.
- Với `n = 15`: `32 768` chuỗi, mọi cách chạy dưới một giây.

## 12. Bộ test

Bài có **nhiều đáp án thứ tự khác nhau**, nên mỗi test: (1) **sắp xếp** rồi so với oracle `generate_bitmask`; (2) kiểm tra **tính chất**: đúng `2ⁿ` chuỗi, **đôi một khác nhau**, mỗi chuỗi là **viết tắt hợp lệ** của `word` (qua `is_valid_abbr`, cài độc lập với mọi cách sinh).

In [13]:
impls = {
    "bitmask": generate_bitmask,
    "backtrack": generate_backtrack,
    "suffix": generate_suffix,
    "iterative": generate_iterative,
    "divide": generate_divide,
    "submit": lambda w: Solution().generateAbbreviations(w),
}


def check_properties(word, result):
    assert len(result) == 2 ** len(word), (word, len(result))
    assert len(set(result)) == len(result), word
    assert all(is_valid_abbr(word, a) for a in result), word


def check(word, expected=None, names=None):
    base = sorted(generate_bitmask(word))
    if expected is not None:
        assert base == expected, (word, base, expected)
    for name, fn in impls.items():
        if names is not None and name not in names:
            continue
        got = fn(word)
        check_properties(word, got)
        assert sorted(got) == base, (name, word)
    return base


for w, e in examples:
    check(w, e)
print("PASS 2 ví dụ của đề (so sánh danh sách đã sắp + tính chất)")

PASS 2 ví dụ của đề (so sánh danh sách đã sắp + tính chất)


### 12.1. Các trường hợp biên và giá trị tính tay

In [14]:
cases = [
    ("a", ["1", "a"], "một chữ cái"),
    ("ab", ["1b", "2", "a1", "ab"], "hai chữ cái"),
    ("abc", sorted(["3", "2c", "1b1", "1bc", "a2", "a1c", "ab1", "abc"]), "ba chữ cái (tính tay ở mục 4)"),
    ("aa", sorted(["2", "1a", "a1", "aa"]), "hai chữ cái giống nhau vẫn cho 4 chuỗi khác nhau"),
    ("zzz", sorted(["3", "2z", "1z1", "1zz", "z2", "z1z", "zz1", "zzz"]), "ba chữ cái giống nhau"),
    ("word", sorted(example_expected), "ví dụ của đề"),
]
for w, e, why in cases:
    assert check(w, e) == e
    print("PASS", why)
assert len(check("abcdefghijklmno")) == 2 ** 15
print("PASS n = 15 (giới hạn đề): 32768 chuỗi")

PASS một chữ cái
PASS hai chữ cái
PASS ba chữ cái (tính tay ở mục 4)
PASS hai chữ cái giống nhau vẫn cho 4 chuỗi khác nhau
PASS ba chữ cái giống nhau
PASS ví dụ của đề


PASS n = 15 (giới hạn đề): 32768 chuỗi


### 12.2. Tính chất và phản ví dụ

In [15]:
rnd = random.Random(320)

# Tính chất 1: chuỗi toàn bộ viết tắt "n" và chính từ "word" đều có mặt
# Tính chất 2: không chuỗi nào chứa hai số kề nhau được hiểu khác (đọc số tối đa liền nhau cho đúng độ dài từ)
# Tính chất 3: chữ cái giống nhau hay khác nhau không làm đổi CẤU TRÚC (mọi chữ thay bằng 'x' cho cùng tập mặt nạ)
for _ in range(200):
    n = rnd.randint(1, 10)
    word = "".join(rnd.choice("abc") for _ in range(n))
    res = generate_backtrack(word)
    assert str(n) in res and word in res
    # cấu trúc: thay mọi chữ bằng 'x' rồi so đa tập mẫu
    pattern = sorted(generate_backtrack("x" * n))
    shapes = sorted("".join("x" if ch.isalpha() else ch for ch in a) for a in res)
    assert shapes == pattern
    # số chữ bị gộp (tổng các số) có phân bố nhị thức: C(n, k) chuỗi có đúng k chữ bị gộp
    hist = {}
    for a in res:
        merged = sum(int(tok) for tok in re.findall(r"\d+", a))
        hist[merged] = hist.get(merged, 0) + 1
    assert all(hist[k] == comb(n, k) for k in range(n + 1))
print("PASS tính chất: có '4'-dạng và chính từ; cấu trúc độc lập chữ cái; phân bố nhị thức C(n, k)")

# Phản ví dụ: '11' không phải viết tắt hợp lệ của từ 2 chữ (hai số không được kề nhau)
assert not is_valid_abbr("ab", "11") and is_valid_abbr("ab", "2")
print("PASS phản ví dụ: '11' bị loại với từ dài 2")

PASS tính chất: có '4'-dạng và chính từ; cấu trúc độc lập chữ cái; phân bố nhị thức C(n, k)
PASS phản ví dụ: '11' bị loại với từ dài 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) với brute force

In [16]:
rnd = random.Random(2026)
for trial in range(300):
    n = rnd.randint(1, 10)
    word = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz"[:rnd.randint(1, 26)]) for _ in range(n))
    check(word)

for trial in range(6):                                    # gần giới hạn đề: n = 13..15
    n = rnd.randint(13, 15)
    word = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    check(word)
print("PASS 300 từ ngẫu nhiên n <= 10 và 6 từ n = 13..15 (mọi cách khớp oracle + tính chất)")

PASS 300 từ ngẫu nhiên n <= 10 và 6 từ n = 13..15 (mọi cách khớp oracle + tính chất)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên chốt số đếm ở cuối từ

In [17]:
def sai_quen_chot_cuoi(word):
    n = len(word)
    res = []

    def dfs(i, cur, cnt):
        if i == n:
            res.append(cur)                                   # sai: thiếu số đếm cuối
            return
        dfs(i + 1, cur, cnt + 1)
        dfs(i + 1, cur + (str(cnt) if cnt else "") + word[i], 0)

    dfs(0, "", 0)
    return res


print("thiếu chốt cuối, 'a' ->", sai_quen_chot_cuoi("a"), "| đúng: ['1', 'a']")
assert sorted(sai_quen_chot_cuoi("a")) != ["1", "a"]

thiếu chốt cuối, 'a' -> ['', 'a'] | đúng: ['1', 'a']


**Vì sao sai:** nhánh "gộp hết đến cuối từ" để lại `count > 0` chưa ghi vào chuỗi, nên được `""` thay cho `"1"`. Phải chốt `count` ở điều kiện dừng.

### Lỗi 2: Ghi số mỗi lần gộp (sinh hai số kề nhau)

In [18]:
def sai_ghi_so_moi_lan(word):
    n = len(word)
    res = []

    def dfs(i, cur):
        if i == n:
            res.append(cur)
            return
        dfs(i + 1, cur + "1")                                  # sai: mỗi chữ gộp ghi "1", cho "11" thay vì "2"
        dfs(i + 1, cur + word[i])

    dfs(0, "")
    return res


print("ghi số mỗi lần, 'ab' ->", sorted(sai_ghi_so_moi_lan("ab")), "| đúng:", ["1b", "2", "a1", "ab"])
assert sorted(sai_ghi_so_moi_lan("ab")) != ["1b", "2", "a1", "ab"]

ghi số mỗi lần, 'ab' -> ['11', '1b', 'a1', 'ab'] | đúng: ['1b', '2', 'a1', 'ab']


**Vì sao sai:** hai chữ liên tiếp bị gộp phải cho **một** số `2`, không phải `"11"`. Cần bộ đếm `count` chỉ chốt khi gặp chữ được giữ hoặc hết từ.

### Lỗi 3: Không chốt số đếm trước khi ghi chữ được giữ

In [19]:
def sai_khong_chot_truoc_chu(word):
    n = len(word)
    res = []

    def dfs(i, cur, cnt):
        if i == n:
            res.append(cur + (str(cnt) if cnt else ""))
            return
        dfs(i + 1, cur, cnt + 1)
        dfs(i + 1, cur + word[i], 0)                          # sai: quên chèn str(cnt) trước chữ được giữ

    dfs(0, "", 0)
    return res


print("không chốt trước chữ, 'ab' ->", sorted(sai_khong_chot_truoc_chu("ab")), "| đúng:", ["1b", "2", "a1", "ab"])
assert sorted(sai_khong_chot_truoc_chu("ab")) != ["1b", "2", "a1", "ab"]

không chốt trước chữ, 'ab' -> ['2', 'a1', 'ab', 'b'] | đúng: ['1b', '2', 'a1', 'ab']


**Vì sao sai:** khi gặp chữ được giữ mà `count > 0`, đoạn bị gộp **phía trước chữ đó** phải được ghi thành số (`"1b"`); nếu quên, số bị mất và còn `"b"`.

### Lỗi 4: Sai biên của vòng lặp bitmask

In [20]:
def sai_bien_bitmask(word):
    return [abbreviation_from_mask(word, mask) for mask in range((1 << len(word)) - 1)]     # sai: thiếu mask cuối (mọi bit 1)


print("thiếu mask cuối, 'a' ->", sai_bien_bitmask("a"), "| đúng: ['a', '1'] (hai chuỗi)")
assert len(sai_bien_bitmask("a")) != 2 and len(sai_bien_bitmask("word")) != 16

thiếu mask cuối, 'a' -> ['a'] | đúng: ['a', '1'] (hai chuỗi)


**Vì sao sai:** có `2ⁿ` mặt nạ từ `0` đến `2ⁿ − 1` (**bao gồm** `2ⁿ − 1`). `range(1 << n)` đúng; `range((1 << n) - 1)` bỏ mất viết tắt toàn bộ (`"4"` với `"word"`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Mỗi phần tử hai lựa chọn (giữ / bỏ)" → **`2ⁿ` kết quả**, dùng **bitmask** hoặc **quay lui** nhị phân.
- Quay lui kèm **bộ đếm "đang gộp"**: chốt số khi gặp chữ được giữ hoặc khi hết từ.
- Hai số **không kề nhau**: tự động đúng nếu chỉ chốt số trước chữ cái hoặc ở cuối.
- Kiểm thử: **sắp xếp** kết quả rồi so; thêm kiểm tra **đúng `2ⁿ`**, **khác nhau từng đôi**, **mỗi chuỗi hợp lệ** (bài 408).
- Giới hạn `n ≤ 15` gợi ý đáp án là hàm mũ (`2¹⁵ = 32768`).
- Họ hàng: **408 Valid Word Abbreviation**, **411 Minimum Unique Word Abbreviation** (khó), **78 Subsets**, **17 Letter Combinations**.

## 15. Biến thể của bài

1. **Kiểm tra viết tắt hợp lệ** (408).
2. **Đếm** số viết tắt (`2ⁿ`) mà không sinh.
3. **Chỉ các viết tắt có đúng `k` chữ bị gộp.**
4. **Phân bố** số viết tắt theo số chữ bị gộp (hệ số nhị thức).
5. **Sinh lười (generator)** để tiết kiệm bộ nhớ.

In [21]:
def count_abbreviations(word):
    """Biến thể 2."""
    return 2 ** len(word)


def abbreviations_with_k_merged(word, k):
    """Biến thể 3: đúng k chữ bị gộp (chọn tập vị trí bằng combinations)."""
    n = len(word)
    out = []
    for positions in combinations(range(n), k):
        mask = 0
        for p in positions:
            mask |= 1 << p
        out.append(abbreviation_from_mask(word, mask))
    return out


def merged_histogram(n):
    """Biến thể 4: hist[k] = số viết tắt có đúng k chữ bị gộp = C(n, k)."""
    return [comb(n, k) for k in range(n + 1)]


def generate_lazy(word):
    """Biến thể 5: generator, sinh lần lượt mà không giữ cả danh sách."""
    n = len(word)

    def dfs(i, cur, cnt):
        if i == n:
            yield cur + (str(cnt) if cnt else "")
            return
        yield from dfs(i + 1, cur, cnt + 1)
        yield from dfs(i + 1, cur + (str(cnt) if cnt else "") + word[i], 0)

    yield from dfs(0, "", 0)


assert is_valid_abbr("internationalization", "i12iz4n") and not is_valid_abbr("apple", "a2e")
rnd = random.Random(15)
for _ in range(100):
    n = rnd.randint(1, 9)
    word = "".join(rnd.choice("abcd") for _ in range(n))
    full = sorted(generate_backtrack(word))
    assert count_abbreviations(word) == len(full)
    assert sorted(generate_lazy(word)) == full
    for k in range(n + 1):
        sub = abbreviations_with_k_merged(word, k)
        assert len(sub) == merged_histogram(n)[k]
        assert all(a in full for a in sub)
assert next(generate_lazy("hello")) == "5"
print("PASS biến thể: hợp lệ 408, đếm, đúng k chữ gộp, phân bố nhị thức, generator")

PASS biến thể: hợp lệ 408, đếm, đúng k chữ gộp, phân bố nhị thức, generator


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (ngoài đầu ra) | Khi nào dùng |
|---|---|---|---|---|
| 1. Bitmask | duyệt `mask`, dựng chuỗi | `O(n·2ⁿ)` | `O(n)` | oracle, dễ viết |
| 2. Quay lui `(pos, cur, count)` | gộp hoặc giữ, chốt số | `O(n·2ⁿ)` | `O(n)` | **cách nộp bài** |
| 3. Đệ quy phần đuôi | chọn độ dài đoạn gộp rồi giữ 1 chữ | `O(n·2ⁿ)` | `O(n·2ⁿ)` | gần lời giải tham khảo |
| 4. Sinh lặp | danh sách `(chuỗi, đếm)` | `O(n·2ⁿ)` | `O(n·2ⁿ)` | không đệ quy |
| 5. Chia để trị | ghép hai nửa, cộng số kề nhau | `O(n·2ⁿ)` | `O(n·2ⁿ)` | minh họa chia để trị |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `sorted(generate_backtrack("ab"))` và `len(generate_backtrack("abcd"))`.

**Bài 2 (tính tay).** Liệt kê mọi viết tắt của `"abc"` và cho biết mỗi viết tắt ứng với `mask` nào.

**Bài 3 (điền TODO).** Hoàn thành `generate_todo` theo Cách 2 (quay lui).

In [22]:
def generate_todo(word):
    # TODO 1: n = len(word); res = []
    # TODO 2: định nghĩa dfs(i, cur, cnt): nếu i == n thì thêm cur + (str(cnt) nếu cnt > 0)
    # TODO 3: nhánh gộp: dfs(i + 1, cur, cnt + 1)
    # TODO 4: nhánh giữ: dfs(i + 1, cur + (str(cnt) nếu cnt) + word[i], 0)
    # TODO 5: gọi dfs(0, "", 0) rồi trả res
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `generate_bitmask2(word)` bằng `mask` (Cách 1) nhưng dựng chuỗi bằng cách duyệt các bit và dùng `itertools.groupby` để gom đoạn bit 1 liên tiếp.

**Bài 5 (biến thể / mini challenge).** (a) Viết `is_valid_abbr2(word, abbr)` (408) bằng biểu thức chính quy hoặc quét hai con trỏ. (b) Viết `shortest_abbreviation(word)` trả về viết tắt **ngắn nhất** (về độ dài chuỗi) khác `"n"` nếu có thể, hoặc cho biết độ dài ngắn nhất tổng quát.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: số kết quả là `2ⁿ`. Bài 2: mỗi bit 1 là chữ bị gộp, các bit 1 liền nhau gộp thành một số.
- **Mức 2 (vừa):** Bài 1: `["1b", "2", "a1", "ab"]` và `16`. Bài 2: bảng ở mục 4. Bài 3: năm TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `groupby(range(n), key=lambda i: (mask >> i) & 1)`; nhóm có key `1` thành `str(len(nhóm))`, nhóm key `0` ghi các chữ. Bài 5(a): duyệt `abbr` bằng hai con trỏ, từ chối số bắt đầu bằng `0`. Bài 5(b): chuỗi `str(n)` luôn có độ dài `len(str(n))` (rất ngắn); nếu loại `"n"` thì chọn trong các viết tắt còn lại.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
from itertools import groupby

# Bài 1
assert sorted(generate_backtrack("ab")) == ["1b", "2", "a1", "ab"]
assert len(generate_backtrack("abcd")) == 16

# Bài 2
assert {abbreviation_from_mask("abc", m): m for m in range(8)} == {
    "abc": 0, "1bc": 1, "a1c": 2, "2c": 3, "ab1": 4, "1b1": 5, "a2": 6, "3": 7}

# Bài 3
def generate_todo_ans(word):
    n = len(word)
    res = []

    def dfs(i, cur, cnt):
        if i == n:
            res.append(cur + (str(cnt) if cnt else ""))
            return
        dfs(i + 1, cur, cnt + 1)
        dfs(i + 1, cur + (str(cnt) if cnt else "") + word[i], 0)

    dfs(0, "", 0)
    return res


# Bài 4
def generate_bitmask2(word):
    n = len(word)
    result = []
    for mask in range(1 << n):
        parts = []
        for bit, group in groupby(range(n), key=lambda i: (mask >> i) & 1):
            idxs = list(group)
            parts.append(str(len(idxs)) if bit else "".join(word[i] for i in idxs))
        result.append("".join(parts))
    return result


# Bài 5
def is_valid_abbr2(word, abbr):
    tokens = re.findall(r"\d+|[a-z]", abbr)
    if "".join(tokens) != abbr:
        return False
    i = 0
    for tok in tokens:
        if tok.isdigit():
            if tok[0] == "0":
                return False
            i += int(tok)
        else:
            if i >= len(word) or word[i] != tok:
                return False
            i += 1
    return i == len(word)


def shortest_abbreviation(word):
    """Độ dài chuỗi ngắn nhất trong các viết tắt (đạt ở 'n' = str(len(word)))."""
    return min(len(a) for a in generate_backtrack(word))


for n in range(1, 9):
    word = "abcdefgh"[:n]
    assert sorted(generate_todo_ans(word)) == sorted(generate_bitmask2(word)) == sorted(generate_bitmask(word))
    assert shortest_abbreviation(word) == len(str(n))
for w in ("word", "abc", "aa"):
    for a in generate_backtrack(w):
        assert is_valid_abbr2(w, a) and is_valid_abbr(w, a)
for a in ("11", "w3d", "wo01", "5", "w2", "1o1d"):
    assert is_valid_abbr2("word", a) == is_valid_abbr("word", a), a
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quay lui với bộ đếm)

```python
class Solution:
    def generateAbbreviations(self, word: str) -> List[str]:
        n = len(word)
        res = []

        def dfs(i, cur, cnt):
            if i == n:
                res.append(cur + (str(cnt) if cnt else ""))
                return
            dfs(i + 1, cur, cnt + 1)                                    # gộp word[i]
            dfs(i + 1, cur + (str(cnt) if cnt else "") + word[i], 0)    # giữ word[i]

        dfs(0, "", 0)
        return res
```

**Checklist**

- [ ] Mỗi vị trí đi **cả hai nhánh** (gộp / giữ), tổng `2ⁿ` kết quả?
- [ ] Chốt số đếm **trước chữ được giữ** và **ở cuối từ** (`cnt > 0`)?
- [ ] Đặt lại `cnt = 0` sau khi chốt?
- [ ] Không có hai số kề nhau và không có số bắt đầu bằng `0`?
- [ ] Thứ tự kết quả tùy ý: khi test đã **sắp xếp** rồi mới so?
- [ ] Đã kiểm tra đúng `2ⁿ` chuỗi, đôi một khác nhau, mỗi chuỗi hợp lệ?
- [ ] Đã thử `n = 1`, chữ cái lặp, `n = 15`?
- [ ] Skeleton đúng `class Solution` / `generateAbbreviations(self, word: str) -> List[str]`?